# 0. Initialization

## Installations

In [ ]:
!pip install torch-fidelity

In [ ]:
!pip install torchmetrics

## Imports

In [ ]:
import os
import sys
import random
from pathlib import Path
from collections import Counter
import csv
import importlib

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt
from torchmetrics.image import StructuralSimilarityIndexMeasure
import torch.nn.functional as F
from tqdm import tqdm
# -------------------------------
# Google Drive
# -------------------------------
from google.colab import drive

drive.mount('/content/drive')
pasta_projeto = "/content/drive/MyDrive/IAG_Project1"
os.chdir(pasta_projeto)
sys.path.append(pasta_projeto)

# -------------------------------
# Python Files
# -------------------------------

# ===================== Data ===================== #
from src.a_Data.artbench_local_dataset import load_kaggle_artbench10_splits
from src.a_Data.load_data import *
from src.a_Data.export_samples import *

# ===================== Train ===================== #
import src.b_Train.models.autoencoders.model_betaVAE as bvae
import src.b_Train.models.autoencoders.model_cVAE as cvae
import src.b_Train.models.autoencoders.model_vqVAE as vqvae
import src.b_Train.models.diffusion.model_diff as diff
#import src.b_Train.models.diffusion.model_diff_vae as diff_vae

import src.b_Train.tune_train.autoencoders.train_betaVAE as train_bvae
import src.b_Train.tune_train.autoencoders.train_cVAE as train_cvae
import src.b_Train.tune_train.autoencoders.tune_bVAE_cVAE as tune_bvae_cvae
import src.b_Train.tune_train.autoencoders.train_vqVAE as train_vqvae
import src.b_Train.tune_train.autoencoders.train_vqVAE_transformer as train_vqvae_transf

import src.b_Train.tune_train.autoencoders.tune_vqVAE as tune_vqvae
import src.b_Train.tune_train.diffusion.train_diff as train_diff
import src.b_Train.tune_train.diffusion.tune_diff as tune_diff
#import src.b_Train.tune_train.diffusion.train_diff_vae as train_diff_vae
#import src.b_Train.tune_train.diffusion.tune_diff_vae as tune_diff_vae
import src.b_Train.tune_train.diffusion.train_spatial_vae as train_spatial_vae

importlib.reload(bvae)
importlib.reload(cvae)
importlib.reload(vqvae)
importlib.reload(diff)
#importlib.reload(diff_vae)

importlib.reload(train_bvae)
importlib.reload(train_cvae)
importlib.reload(tune_bvae_cvae)
importlib.reload(train_vqvae)
importlib.reload(train_vqvae_transf)
importlib.reload(tune_vqvae)
importlib.reload(train_diff)
importlib.reload(tune_diff)
#importlib.reload(train_diff_vae)
#importlib.reload(tune_diff_vae)
importlib.reload(train_spatial_vae)

from src.b_Train.models.autoencoders.model_betaVAE import *
from src.b_Train.models.autoencoders.model_cVAE import *
from src.b_Train.models.autoencoders.model_vqVAE import *
from src.b_Train.models.diffusion.model_diff import *
#from src.b_Train.models.diffusion.model_diff_vae import *

from src.b_Train.tune_train.autoencoders.train_betaVAE import *
from src.b_Train.tune_train.autoencoders.train_cVAE import *
from src.b_Train.tune_train.autoencoders.tune_bVAE_cVAE import *
from src.b_Train.tune_train.autoencoders.train_vqVAE import *
from src.b_Train.tune_train.autoencoders.train_vqVAE_transformer import *
from src.b_Train.tune_train.autoencoders.tune_vqVAE import *
from src.b_Train.tune_train.diffusion.train_diff import *
from src.b_Train.tune_train.diffusion.tune_diff import *
#from src.b_Train.tune_train.diffusion.train_diff_vae import *
#from src.b_Train.tune_train.diffusion.tune_diff_vae import *
from src.b_Train.tune_train.diffusion.train_spatial_vae import *

# ===================== Evaluation ===================== #
import src.c_Evaluation.sample_models as sample
import src.c_Evaluation.evaluate_models as evaluate

importlib.reload(sample)
importlib.reload(evaluate)

from src.c_Evaluation.sample_models import *
from src.c_Evaluation.evaluate_models import *

# ===================== Utils ===================== #
import src.utils.plots as plots

importlib.reload(plots)

from src.utils.plots import *


# 1. Data Loading

### Config

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
KAGGLE_ROOT = Path('datasets/archive')
TRAINING_CSV_PATH = Path('datasets/training_20_percent.csv')
INDEX_COLUMN = 'train_id_original'

# Parameters
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 2
EFFECTIVE_NUM_WORKERS = safe_num_workers(NUM_WORKERS)

transform = T.Compose([
    T.Resize(IMAGE_SIZE, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),  # normaliza para [0,1]
])



In [ ]:
# Load
hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
train_hf = hf_ds["train"]
test_hf = hf_ds["test"]

print("Train size:", len(train_hf))
print("Test size:", len(test_hf))
print("Columns   :", train_hf.column_names)

label_feature = train_hf.features["label"]
class_names = list(label_feature.names)
num_classes = len(class_names)
print("Num classes:", num_classes)
print("Class names:", class_names)


In [ ]:
# Class distribution summary
train_counts = Counter(train_hf["label"])
test_counts = Counter(test_hf["label"])

print("\nTrain class distribution:")
for cid, name in enumerate(class_names):
    print(f"  {cid:2d} | {name:>15s} | {train_counts.get(cid, 0):6d}")


print("\nTest class distribution:")
for cid, name in enumerate(class_names):
    print(f"  {cid:2d} | {name:>15s} | {test_counts.get(cid, 0):6d}")

## Full load (Used to train the best models)

In [ ]:
VAL_FRACTION = 0.2
all_train_ids = list(range(len(train_hf)))
num_val = int(len(all_train_ids) * VAL_FRACTION)
val_indices = all_train_ids[:num_val]
train_indices = all_train_ids[num_val:]

# Datasets HFDatasetTorch
full_train_ds = HFDatasetTorch(train_hf, transform=transform, indices=train_indices)
full_val_ds   = HFDatasetTorch(train_hf, transform=transform, indices=val_indices)
full_test_ds  = HFDatasetTorch(test_hf, transform=transform)  # test completo

# DataLoaders
full_train_loader = DataLoader(
    full_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_val_loader = DataLoader(
    full_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

full_test_loader = DataLoader(
    full_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)


print("Train dataset length (after fraction):", len(full_train_ds))
print("Train batches                        :", len(full_train_loader))

print("Val dataset length (after fraction):", len(full_val_ds))
print("Val batches                        :", len(full_val_loader))

print("Test dataset length (after fraction):", len(full_test_ds))
print("Test batches                        :", len(full_test_loader))

## Fixed loading (20% csv file)

In [ ]:
train_ids = load_ids_from_training_csv(TRAINING_CSV_PATH, index_column=INDEX_COLUMN)
VAL_FRACTION = 0.2
num_val = int(len(train_ids) * VAL_FRACTION)
train_indices = train_ids[num_val:]
val_indices = train_ids[:num_val]

# Datasets HFDatasetTorch
train_ds = HFDatasetTorch(train_hf, transform=transform, indices=train_indices)
val_ds = HFDatasetTorch(train_hf, transform=transform, indices=val_indices)

# Dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=EFFECTIVE_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('Subset train dataset length:', len(train_ds))
print('Subset train batches      :', len(train_loader))

print('Subset val dataset length:', len(val_ds))
print('Subset val batches      :', len(val_loader))



## Pre processing

In [ ]:
sample = train_ds[0]

img = sample[0]
label = sample[1]
idx = sample[2]
print(img.shape)

In [ ]:
labels_train = [train_ds[i][1] for i in range(len(train_ds))]
labels_val = [val_ds[i][1] for i in range(len(val_ds))]

class_counts_train = Counter(labels_train)
class_counts_val = Counter(labels_val)

print("Distribuição de classes (Train):", class_counts_train)
print("Distribuição de classes (Val):", class_counts_val)

In [ ]:
fixed_batch = next(iter(full_val_loader))

if isinstance(fixed_batch, (list, tuple)):
    fixed_images = fixed_batch[0]
    fixed_labels = fixed_batch[1] if len(fixed_batch) > 1 else None
else:
    fixed_images = fixed_batch
    fixed_labels = None

## Plot

In [ ]:
def show_batch_grid(loader, class_names, n_images=36, nrow=6, title='Sample Grid'):
    x, y, idx = next(iter(loader))
    x = x[:n_images]
    y = y[:n_images]

    grid = make_grid(x, nrow=nrow, padding=2)
    np_img = grid.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(np_img)
    plt.axis('off')
    plt.title(title)
    plt.show()

    # Print labels for quick inspection
    labels_str = [class_names[int(v)] for v in y]
    print('Labels:', labels_str)


show_batch_grid(train_loader, class_names, n_images=36, nrow=6, title='ArtBench-10 Train Samples')

# 2. Training

## 2.1. Autoencoders

### **Beta-VAE**

#### Hyperparameter Tuning

In [ ]:
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
model_params, train_params, best_score, summary = tuning_bvae_cvae(BetaVAE, train_loader, val_loader, beta_vae_train, ssim_metric,num_epochs=100, device='cuda')

In [ ]:
model_params, train_params,

#### Training with the Optimal Configuration

In [ ]:
model = BetaVAE(color_channels=3, latent_dim=256, hidden_dim=64, input_dim=32)

trained_model, outputs, best_val_loss, best_epoch, val_losses = beta_vae_train(model=model, train_dataloader=full_train_loader,  val_dataloader=full_val_loader,ssim_fn=ssim_metric,
                                                                               num_epochs=120, learnR=1e-4, beta_start=0.0, beta_max=0.05, warmup_epochs=20, patience=10, min_delta=0.1,
                                                                                device='cuda', loss_type="mse")

In [ ]:
torch.save(trained_model.state_dict(), "/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_betavae.pt")

#### Reconstruction and Generated Images Visualization

In [ ]:
bvae_model = BetaVAE(color_channels=3, latent_dim=256, hidden_dim=64, input_dim=32, loss_type="mse")
bvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_betavae.pt"))

generated_images = generate_fake_imgs_bvae(bvae_model, latent_dim=256, batch_size=64, num_samples=8, device='cuda')
plot_reconstruction_and_generation(bvae_model, fixed_images, generated_images, device='cuda', num_images=8)

### **Condicional Beta-VAE**

#### Hyperparameter Tuning

In [ ]:
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
model_params, train_params, best_score, summary = tuning_bvae_cvae(CVAE, train_loader, val_loader, cvae_train, ssim_metric,num_epochs=60, device='cuda')

In [ ]:
model_params, train_params

#### Training with the Optimal Configuration

In [ ]:
model = CVAE(color_channels=3, latent_dim=256, hidden_dim=64, input_dim=32)

trained_model, outputs, best_val_loss, best_epoch, val_losses = cvae_train(model=model, train_dataloader=full_train_loader,  val_dataloader=full_val_loader,ssim_fn=ssim_metric,
                                                                               num_epochs=120, learnR=1e-4, beta_start=0.0, beta_max=0.05, warmup_epochs=20, patience=10, min_delta=0.1,
                                                                                device='cuda', loss_type="mse")

In [ ]:
torch.save(trained_model.state_dict(), "/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_cvae.pt")

####  Reconstruction and Generated Images Visualization

In [ ]:
cvae_model = CVAE(color_channels=3, latent_dim=256, hidden_dim=64, input_dim=32)
cvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_cvae.pt"))

generated_images = generate_fake_imgs_cvae(cvae_model, latent_dim=256, batch_size=64, num_samples=8, device='cuda')
plot_reconstruction_and_generation(cvae_model, fixed_images, generated_images, device='cuda', num_images=8,labels = fixed_labels)

### **Vector Quantized-VAE**

#### Hyperparameter Tuning

In [ ]:
model_params, train_params, best_score, summary = tuning_vqvae_phase(VQVAE, train_loader, val_loader,
                                                                     vqvae_train, num_epochs=100, device='cuda')

In [ ]:
model_params, train_params

#### Training with the Optimal Configuration

In [ ]:
# model
in_channels = 3
decay = 0.95
hidden_dim = 128
commitment_cost = 0.25
diversity_lambda = 0.0
num_embeddings = 128
embedding_dim = 128

# train
learnR = 5e-5
loss_type = "mse"
num_epochs=120
patience=20
min_delta=1e-5
device='cuda'
model = VQVAE(in_channels, hidden_dim, embedding_dim, num_embeddings, commitment_cost, decay, diversity_lambda)
model, _, _, _, _, _ = vqvae_train(model, full_train_loader, full_val_loader, num_epochs, learnR, patience, min_delta, device, loss_type)

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_vqvae.pt")

 #### Reconstruction and Generated Images Visualization

In [ ]:
transformer_model, codebook_shape = transformer
generated_images = generate_fake_imgs_vqvae(model, transformer_model, codebook_shape, num_samples=8, device="cuda")

In [ ]:
plot_reconstruction_and_generation(model, fixed_images, generated_images, device='cuda', num_images=8)

## 2.2. Diffusion Models

### **DDPM**

#### Hyperparameter Tuning

In [ ]:
model = PixelUNet2(in_channels=3, model_channels=64).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

model_diff, ema_diff, train_losses, val_losses = train_diffusion(model, diffusion, train_loader, val_loader, optimizer, ema, device, epochs=60)

In [ ]:
model = PixelUNet3(in_channels=3, model_channels=64).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

model_diff, ema_diff, train_losses, val_losses = train_diffusion(model, diffusion, train_loader, val_loader, optimizer, ema, device, epochs=60)

In [ ]:
model_params, best_lr, _, _ = tuning_diffusion_phases(train_loader, val_loader, device, depth=3, num_epochs=60)

In [ ]:
model_params, best_lr

#### Training with the Optimal Configuration

In [ ]:
model = PixelUNet3(in_channels=3, model_channels=128, gn_groups=8).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

model_diff, ema_diff, train_losses, val_losses = train_diffusion(model, diffusion, full_train_loader, full_val_loader, optimizer, ema, device, epochs=120)

In [ ]:
torch.save({
    'model_diff': model_diff.state_dict(),
    'ema_diff': ema_diff.ema_model.state_dict()
}, '/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_diff.pt')

#### Generated Images

In [ ]:
model = PixelUNet3(in_channels=3, model_channels=128, gn_groups=8).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
caminho = '/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_diff.pt'
checkpoint = torch.load(caminho, map_location=device)

model.load_state_dict(checkpoint['model_diff'])
ema.ema_model.load_state_dict(checkpoint['ema_diff'])

In [ ]:
fake_images = diffusion.sample_ddim(model, ema)
plot_diffusion_samples(real_images, fake_images, device='cuda', num_images=8)

### **Latent Diffusion**

#### Hyperparameter Tuning

In [ ]:
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
model = SpatialBetaVAE(color_channels=3, latent_channels=128, hidden_dim=128, beta=0.05, loss_type="mse")
model, _,_,_ = spatial_beta_vae_train(model, train_loader, val_loader, ssim_fn=ssim_metric, num_epochs=100, learnR=1e-4,
                       beta_start=0.0,  beta_max=0.05,  warmup_epochs=20,  patience=10,  min_delta=1e-3,
                       device='cuda', loss_type="mse")

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_spacial_betavae.pt")

In [ ]:
spatialvae_model = SpatialBetaVAE(color_channels=3, latent_channels=128, hidden_dim=128, beta=0.05, loss_type="mse")
spatialvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_spacial_betavae.pt"))
spatialvae_model.to(device="cuda")
spatialvae_model.eval()
for p in spatialvae_model.parameters():
    p.requires_grad = False

model = PixelUNet2(in_channels=128, model_channels=64).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
model_diff, ema_diff,_,_ = train_diffusion_vae(model, diffusion, spatialvae_model, full_train_loader, full_val_loader, optimizer, ema, device="cuda", epochs=40)

In [ ]:
spatialvae_model = SpatialBetaVAE(color_channels=3, latent_channels=128, hidden_dim=128, beta=0.05, loss_type="mse")
spatialvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_spacial_betavae.pt"))
spatialvae_model.to(device="cuda")
spatialvae_model.eval()
for p in spatialvae_model.parameters():
    p.requires_grad = False

model = PixelUNet3(in_channels=128, model_channels=64).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
model_diff, ema_diff,_,_ = train_diffusion_vae(model, diffusion, spatialvae_model, full_train_loader, full_val_loader, optimizer, ema, device="cuda", epochs=40)

In [ ]:
model_params, best_lr,_,_ = tuning_ldm_phases(train_loader, val_loader, spatialvae_model, device="cuda", depth=2, num_epochs=40)

In [ ]:
model_params, best_lr

#### Training with the Optimal Configuration

In [ ]:
spatialvae_model = SpatialBetaVAE(color_channels=3, latent_channels=128, hidden_dim=128, beta=0.05)
spatialvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_spacial_betavae.pt"))
spatialvae_model.to(device="cuda")
spatialvae_model.eval()
for p in spatialvae_model.parameters():
    p.requires_grad = False

model = PixelUNet2(in_channels=128, model_channels=128, gn_groups=8).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
model_diff, ema_diff,_,_ = train_diffusion_vae(model, diffusion, spatialvae_model, full_train_loader, full_val_loader, optimizer, ema, device="cuda", epochs=120)

In [ ]:
torch.save({
    'model_diff_vae': model_diff.state_dict(),
    'ema_diff_vae': ema_diff.ema_model.state_dict()
}, '/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_diff_vae.pt')

#### Generated Images

In [ ]:
model = PixelUNet2(in_channels=128, model_channels=128, gn_groups=8).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
caminho = '/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_diff_vae.pt'
checkpoint = torch.load(caminho, map_location=device)

model.load_state_dict(checkpoint['model_diff_vae'])
ema.ema_model.load_state_dict(checkpoint['ema_diff_vae'])

spatialvae_model = SpatialBetaVAE(color_channels=3, latent_channels=128, hidden_dim=128, beta=0.05, loss_type="mse")
spatialvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_spacial_betavae.pt"))
spatialvae_model.to(device="cuda")
spatialvae_model.eval()
for p in spatialvae_model.parameters():
    p.requires_grad = False

In [ ]:
fake_images = sample_ldm(model, diffusion, spatialvae_model, ema, n=8)
plot_diffusion_samples(real_images, fake_images, device='cuda', num_images=8)

# 3. Evaluation

## Sampling

In [ ]:
transform = T.Compose([
    T.Resize(32),
    T.CenterCrop(32),
    T.ToTensor()  # [0,1]
])

real_images = sample_real_images(
    dataset=test_hf,
    indices=range(len(test_hf)),
    num_per_class=500,
    num_classes=10,
    transform=transform
)

## Fid/Kid Evaluation

In [ ]:
num_samples = 5000
num_runs = 10

### Autoencoders

BetaVAE

In [ ]:
bvae_model = BetaVAE(color_channels=3, latent_dim=256, hidden_dim=64, input_dim=32)
bvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_betavae.pt"))
params = (bvae_model, 256, 64)

evaluate_fid_kid("bvae", params, real_images, num_samples, num_runs, device='cuda')

CVAE

In [ ]:
cvae_model = CVAE(color_channels=3, latent_dim=256, hidden_dim=64, input_dim=32)
cvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_cvae.pt"))
params = (cvae_model, 256, 64)

evaluate_fid_kid("cvae", params, real_images, num_samples, num_runs, device='cuda')

VQVAE

In [ ]:
vqvae_model = VQVAE(in_channels=3, hidden_dim=128, embedding_dim=128, num_embeddings=128, commitment_cost=0.25, decay=0.95, diversity_lambda=0.0)
vqvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_vqvae.pt"))

transformer = train_vqvae_transformer(vqvae_model, train_loader, batch_size=64, device='cuda')

In [ ]:
model = VQVAE(in_channels=3, hidden_dim=128, embedding_dim=128, num_embeddings=128, commitment_cost=0.25, decay=0.95, diversity_lambda=0.0)
model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_vqvae.pt"))
transformer_model, codebook_shape = transformer
params = (model, transformer_model, codebook_shape)
evaluate_fid_kid("vqvae", params, real_images, num_samples, num_runs, device='cuda')

### Difussion Models

DDPM

In [ ]:
model = PixelUNet3(in_channels=3, model_channels=128, gn_groups=8).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
caminho = '/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_diff.pt'
checkpoint = torch.load(caminho, map_location=device)
model.load_state_dict(checkpoint['model_diff'])
ema.ema_model.load_state_dict(checkpoint['ema_diff'])

params = (model, diffusion, ema)
evaluate_fid_kid("diff", params, real_images, num_samples, num_runs, device='cuda')

LDM

In [ ]:
model = PixelUNet2(in_channels=128, model_channels=128, gn_groups=8).to(device)
diffusion = GaussianDiffusion(device=device)
ema = EMA(model, decay=0.9999)
ema.to(device)
caminho = '/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_diff_vae.pt'
checkpoint = torch.load(caminho, map_location=device)

model.load_state_dict(checkpoint['model_diff_vae'])
ema.ema_model.load_state_dict(checkpoint['ema_diff_vae'])

spatialvae_model = SpatialBetaVAE(color_channels=3, latent_channels=128, hidden_dim=128, beta=0.05, loss_type="mse")
spatialvae_model.load_state_dict(torch.load("/content/drive/MyDrive/IAG_Project1/src/b_Train/saved_models/saved_spacial_betavae.pt"))
spatialvae_model.to(device="cuda")
spatialvae_model.eval()
for p in spatialvae_model.parameters():
    p.requires_grad = False

params = (model, diffusion, spatialvae_model, ema)
evaluate_fid_kid("diff_vae", params, real_images, num_samples, num_runs, device='cuda')